## Data Extraction

In [14]:
# import libraries

from datetime import datetime, timedelta, timezone

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
import os
from dotenv import load_dotenv
import pandas as pd

### 15 Minute Stock Data

We'll retrieve 15 Minute price data for TSLA, NVDA, PLTR, COIN, & OKLO

These 5 stocks have heavy options activity and I primarily focus on trading these tickers

In [21]:
load_dotenv()

ALPACA_API_KEY = os.getenv("ALPACA_API_KEY")
ALPACA_SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise ValueError("Missing Alpaca keys. Check your .env file.")

client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

# pull 2 years of TSLA data 15M price data
end = datetime.now(timezone.utc) - timedelta(minutes=20)
start = end - timedelta(days=365 * 2) 

all_dfs = []

current_start = start

while current_start < end:
    current_end = min(current_start + timedelta(days=90), end)

    req = StockBarsRequest(
        symbol_or_symbols="COIN",
        timeframe=TimeFrame(15, TimeFrameUnit.Minute),
        start=current_start,
        end=current_end,
        limit=10000,
    )

    bars = client.get_stock_bars(req).df

    if not bars.empty:
        all_dfs.append(bars)

    print(f"Pulled {current_start.date()} → {current_end.date()} : {len(bars)} rows")

    current_start = current_end

df = pd.concat(all_dfs).reset_index()

print("Total rows:", len(df))
df.head()

Pulled 2024-02-26 → 2024-05-26 : 4033 rows
Pulled 2024-05-26 → 2024-08-24 : 3895 rows
Pulled 2024-08-24 → 2024-11-22 : 4022 rows
Pulled 2024-11-22 → 2025-02-20 : 3652 rows
Pulled 2025-02-20 → 2025-05-21 : 3965 rows
Pulled 2025-05-21 → 2025-08-19 : 3857 rows
Pulled 2025-08-19 → 2025-11-17 : 3977 rows
Pulled 2025-11-17 → 2026-02-15 : 3847 rows
Pulled 2026-02-15 → 2026-02-25 : 415 rows
Total rows: 31663


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,COIN,2024-02-26 17:00:00+00:00,185.590,187.1699,184.7100,186.8700,481497.0,7101.0,186.136366
1,COIN,2024-02-26 17:15:00+00:00,186.805,187.5000,186.3600,186.8200,318263.0,5111.0,186.944828
2,COIN,2024-02-26 17:30:00+00:00,186.715,189.9300,186.6601,189.8000,640690.0,9636.0,188.864781
3,COIN,2024-02-26 17:45:00+00:00,189.880,189.9700,188.1400,189.7300,434089.0,6478.0,189.023027
4,COIN,2024-02-26 18:00:00+00:00,189.750,191.0000,188.2517,189.4401,617964.0,9070.0,189.642273


In [22]:
df.to_csv("COIN_15min_2yr.csv", index=False)
print("Saved COIN_15min_2yr.csv")

Saved COIN_15min_2yr.csv
